# Signatures comparison — TRUE validation cohort (CHESS-1336)

Companion to the published Figure 4 notebook (`Signatures comparison.ipynb`) and to
`Signatures comparison NEW.ipynb`. The new sorted-cell cohort delivered with
[Jira OD-128](https://bostongene.atlassian.net/browse/OD-128) is used here as a
**true held-out validation cohort** — this is the CHESS-1336 rerun of the original
cell-type FGES benchmark on validation data.

**Scope:** the 16 in-scope FGES (`MAP_RAW`). The four rare-GOI FGES
(`Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`,
`Main4_Plasma_cells`, i.e. `EXCLUDED_FGES_RARE`) are deferred to the separate
rare-types notebook (75/25 stratified holdouts, `signature_validation.benchmark.splits`).

**Baseline:** the published Figure-4 scores (`data/mapping_ssgseas.pkl`) are re-scored
through the *same* `plot_sens_spec_scatter` code path, so the validation-vs-baseline
sensitivity/specificity deltas use an identical metric definition on both cohorts.

**Random-FGES baseline:** v1 random gene lists are reused verbatim from
`data/msigdb_gmt.pkl` and only re-scored on the validation cohort, so ranks stay
comparable.

**Outputs:** everything is written under `../plots/` and `../tables/` with a
`_validation` suffix; v1 and NEW outputs are never overwritten.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import s3fs
import seaborn as sns
import zarr
from loguru import logger

from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    MAP_RAW,
    build_mapping,
    intersect_controls_with_cohort,
    load_new_cohort_annotation,
)
from signature_validation.benchmark.plotting import (
    plot_sens_spec_scatter,
    plot_signature_heatmap,
    plot_violin_per_source,
)
from signature_validation.benchmark.scoring import (
    compute_mapping_ssgseas,
    compute_out_table,
    fdr_correct_out,
)
from signature_validation.benchmark.signatures import (
    count_random_fges,
    harmonize_gmt_to_index,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.plotting.plotting import cells_p

sns.set_style("white")
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
# --- Validation-cohort inputs (local copies, committed under data/) ---
VALIDATION_ANNOT_PATH = Path("../data/sorted_cells_to_check_all_annot.tsv")

# Expressions live on S3 as a zarr v3 array (obs x var), NOT per-GSE TSV.
# read_expressions does NOT apply here; §4 reads this zarr directly (see load_osrp_expressions).
OSRP_ZARR = (
    "bostongene-eurynome-exchange/raw_data/v2/expressions/osrp_tier1_expressions.zarr"
)

# --- Baseline / signature inputs (committed in the repo) ---
V1_GMT_PICKLE = Path("../data/msigdb_gmt.pkl")              # real local pickle
BASELINE_SSGSEAS_PATH = Path("../data/mapping_ssgseas.pkl")  # real local pickle (199 MB)

# --- Outputs (repo folders, `_validation` suffix; v1/NEW never overwritten) ---
OUTPUT_DIR = Path("../plots")
TABLES_DIR = Path("../tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_SSGSEAS_PATH = OUTPUT_DIR / "mapping_ssgseas_validation.pkl"
OUT_TSV_PATH = TABLES_DIR / "out_validation.tsv"
CMP_TSV_PATH = TABLES_DIR / "comparison_validation_vs_baseline.tsv"
HEATMAP_PATH = OUTPUT_DIR / "signature_heatmap_validation.svg"

# Fail loudly, early, if a local input is missing.
for label, p in (
    ("validation annotation", VALIDATION_ANNOT_PATH),
    ("v1 GMT pickle", V1_GMT_PICKLE),
    ("baseline ssgseas pickle", BASELINE_SSGSEAS_PATH),
):
    if not p.exists():
        logger.error("{} not found: {}", label, p)

logger.info("outputs -> {} and {}", OUTPUT_DIR, TABLES_DIR)

## §3 — Load validation annotation

> The loader does **not** filter by QC (its docstring assumes a pre-filtered file), so we apply the OD-128 filter explicitly: `Technical_QC == True` AND `Decision_deconvolution_without_parent != False` (30418 -> ~17322 samples).

In [ ]:
validation_annot = load_new_cohort_annotation(VALIDATION_ANNOT_PATH)

# Explicit QC filter (not applied inside load_new_cohort_annotation).
qc_mask = (validation_annot["Technical_QC"] == True) & (
    validation_annot["Decision_deconvolution_without_parent"] != False
)
validation_annot = validation_annot[qc_mask]
logger.info("annotation after QC filter: {} samples", len(validation_annot))
validation_annot["Cell_type"].value_counts()

## §4 — Load validation expressions (from osrp zarr)

> Expressions are a zarr v3 array (obs x var: SRX samples x gene symbols), not the per-GSE TSVs `read_expressions` expects. We read the needed rows straight from S3 (lazy chunks) and return a genes x samples frame. Only samples whose id is in the zarr `obs_names` are returned (BGP internal samples are absent from open-source osrp).

In [ ]:
def load_osrp_expressions(
    sample_ids: list[str],
    zarr_path: str = OSRP_ZARR,
    log2: bool = True,
) -> pd.DataFrame:
    """Load osrp zarr expressions for the given samples as a genes x samples frame.

    Parameters
    ----------
    sample_ids : list[str]
        Sample ids from the annotation index; the intersection with ``obs_names``
        is taken automatically.
    zarr_path : str
        Bucket+key of the zarr array (no ``s3://`` scheme).
    log2 : bool
        Apply ``log2(TPM + 1)`` (v1-pipeline transform). Set False if the array is
        already in log space (check ``expr.max()``: ~15-20 => raw TPM; ~4-5 => log).

    Returns
    -------
    pd.DataFrame
        Genes x samples (index = ``var_names``, columns = matched ``obs_names``).

    Raises
    ------
    KeyError
        If no ``sample_ids`` are present in the zarr ``obs_names``.
    """
    fs = s3fs.S3FileSystem()
    store = zarr.storage.FsspecStore(fs, path=zarr_path)  # zarr v3
    z = zarr.open(store, mode="r")

    obs = list(z.attrs["obs_names"])
    var = list(z.attrs["var_names"])
    pos = {s: i for i, s in enumerate(obs)}

    want = [s for s in sample_ids if s in pos]
    if not want:
        raise KeyError("no sample_ids found in osrp zarr obs_names")
    rows = sorted(pos[s] for s in want)
    logger.info("osrp zarr: {} / {} samples matched", len(rows), len(sample_ids))

    x = z.oindex[rows, :]  # (n_want, n_genes) float32
    expr = pd.DataFrame(x.T, index=var, columns=[obs[r] for r in rows])
    if log2:
        expr = np.log2(expr + 1)
    logger.info("expressions: {} genes x {} samples", expr.shape[0], expr.shape[1])
    return expr


validation_expr = load_osrp_expressions(list(validation_annot.index))
validation_expr.shape

## §5 — Build FGES mapping (16 in-scope FGES, scoped to the validation cohort)

In [ ]:
mapping = build_mapping(annotation=validation_annot)
controls_present = intersect_controls_with_cohort(CONTROLS_ORDER, validation_annot)
logger.info(
    "in-scope FGES: {}; controls present in validation cohort: {}",
    len(mapping),
    len(controls_present),
)
for sign, bucket in mapping.items():
    logger.info(
        "{}: GOI={}, Control={}, Deleted={}",
        sign,
        bucket["Goi"],
        len(bucket["Control"]),
        len(bucket["Deleted_controls"]),
    )

## §6 — Reuse v1 gene lists (byte-identical) and harmonize to the validation index

In [ ]:
v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
in_scope_fges = [k for k in MAP_RAW if k not in EXCLUDED_FGES_RARE]
v1_gmt = select_msigdb_gmt_subset(v1_gmt_full, in_scope_fges)

for sign in in_scope_fges:
    assert sign in v1_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    n_random = count_random_fges(v1_gmt[sign])
    assert n_random == 10, f"{sign}: expected 10 RANDOM_FGES, got {n_random}"

msigdb_gmt = harmonize_gmt_to_index(v1_gmt, validation_expr.index)
logger.info(
    "msigdb_gmt: {} FGES, {} sub-signatures total",
    len(msigdb_gmt),
    sum(len(v) for v in msigdb_gmt.values()),
)

## §7 — Compute ssGSEA on the validation cohort

In [ ]:
mapping_ssgseas = compute_mapping_ssgseas(
    public_cells_expr=validation_expr,
    public_cells_annot=validation_annot,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
)

for sign in mapping_ssgseas:
    if sign in EXCLUDED_FGES_RARE:
        continue
    assert mapping_ssgseas[sign]["Goi"], f"{sign}: GOI cohort is empty"
    for ct, frame in mapping_ssgseas[sign]["Goi"].items():
        logger.info("{} GOI {}: {} samples", sign, ct, frame.shape[0])

with open(MAPPING_SSGSEAS_PATH, "wb") as fh:
    pickle.dump(mapping_ssgseas, fh, pickle.HIGHEST_PROTOCOL)
logger.info("wrote {}", MAPPING_SSGSEAS_PATH)

## §8 — Per-signature × cell-type stats table (+ deterministic FDR)

In [ ]:
out = compute_out_table(mapping_ssgseas, mapping, msigdb_gmt, controls_present)
out = fdr_correct_out(out, controls_present)
out.to_csv(OUT_TSV_PATH, sep="\t")
logger.info("wrote {} ({} rows x {} cols)", OUT_TSV_PATH, *out.shape)
out.head()

> **Checkpoint.** §1–§8 above form the minimal working skeleton and must run top-to-bottom on the BG cluster with no errors before §9–§10.

## §9 — Validation vs published baseline (sensitivity / specificity)

Option A: the published Figure-4 scores (`data/mapping_ssgseas.pkl`) are re-scored
through the same `plot_sens_spec_scatter` code path used for validation, so the delta
uses an identical metric definition on both cohorts. Run `git lfs pull` first — the
baseline pickle is LFS-tracked.

In [ ]:
if not BASELINE_SSGSEAS_PATH.exists():
    raise FileNotFoundError(f"baseline scores not found: {BASELINE_SSGSEAS_PATH}")
with open(BASELINE_SSGSEAS_PATH, "rb") as fh:
    head = fh.read(64)
if head.startswith(b"version https://git-lfs"):
    raise RuntimeError(
        f"{BASELINE_SSGSEAS_PATH} is an unresolved git-LFS pointer; run `git lfs pull`"
    )
with open(BASELINE_SSGSEAS_PATH, "rb") as fh:
    baseline_ssgseas = pickle.load(fh)
logger.info("baseline scores loaded: {} FGES", len(baseline_ssgseas))

In [ ]:
# `val_avg` is produced here and reused by the §10 scatter; `base_avg` uses a
# throwaway suffix (its SVGs are not part of the deliverable).
val_avg = plot_sens_spec_scatter(
    mapping_ssgseas=mapping_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
    suffix="_validation",
)
base_avg = plot_sens_spec_scatter(
    mapping_ssgseas=baseline_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
    suffix="_baseline_tmp",
)

# Remove the throwaway baseline SVGs so only `_validation` artefacts remain.
for tmp_svg in OUTPUT_DIR.glob("*_baseline_tmp*"):
    tmp_svg.unlink()

In [ ]:
rows: list[dict] = []
for sign in val_avg:
    for axis in ("Sensitivity", "Specificity"):
        for src, series in val_avg[sign][axis].items():
            baseline_series = base_avg.get(sign, {}).get(axis, {}).get(src)
            rows.append(
                {
                    "FGES": sign,
                    "axis": axis,
                    "source": src,
                    "validation": float(series.mean()),
                    "baseline": (
                        float(baseline_series.mean())
                        if baseline_series is not None
                        else float("nan")
                    ),
                }
            )
cmp = pd.DataFrame(rows)
cmp["delta"] = cmp["validation"] - cmp["baseline"]
cmp.to_csv(CMP_TSV_PATH, sep="\t", index=False)
logger.info("wrote {} ({} rows)", CMP_TSV_PATH, len(cmp))
cmp.head(20)

## §10 — Figures (suffix `_validation`; rare cell types starred)

The plot helpers already carry the `_validation` suffix and star rare cell types /
rare FGES automatically. The validation sens/spec scatter was produced in §9
(`val_avg`); it is not re-run here.

In [ ]:
plot_violin_per_source(mapping_ssgseas, save_dir=OUTPUT_DIR, suffix="_validation")
plot_signature_heatmap(
    mapping_ssgseas=mapping_ssgseas,
    out_df=out,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
    annotation=validation_annot,
    controls_order=controls_present,
    palette={ct: cells_p[ct] for ct in controls_present if ct in cells_p},
    save_path=HEATMAP_PATH,
    short=True,
)
logger.info("plots saved under {}", OUTPUT_DIR)

## §11 — Rare cell types (out of scope here)

FGES whose GOI is rare in the validation cohort — `Main4_Th17_signature`,
`Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` —
are deferred to a separate rare-types notebook. That notebook reuses the original
cohort, generates 10 stratified 75/25 holdouts via
`signature_validation.benchmark.splits.stratified_holdout_indices` (stratified by
BG-FGES score median × GOI/Control), scores ssGSEA on each test fold and aggregates
with `aggregate_score_over_splits`. Rare cell types are starred on the resulting
figures.